In [1]:
!pip -q install duckdb pyarrow

PARQUET = "df_merge_clean.parquet"  # ajuste se o nome for outro

In [2]:
import duckdb, os

con = duckdb.connect()

# Base: só 2022
con.execute(f"""
    CREATE OR REPLACE TABLE base AS
    SELECT pdv, produto, iso_week, quantity
    FROM read_parquet('{PARQUET}')
    WHERE iso_year = 2022
""")

# Escolha *recente* para reduzir linhas
#  últimos 8 semanas de 2022
#  top-50 SKUs por PDV nesse recorte (ajuste 50 se precisar reduzir mais)
con.execute("""
    CREATE OR REPLACE TABLE recent_top AS
    WITH rec AS (
        SELECT
            pdv, produto,
            SUM(quantity) AS qty8
        FROM base
        WHERE iso_week >= 45
        GROUP BY 1,2
        HAVING SUM(quantity) > 0
    ),
    ranked AS (
        SELECT
            pdv, produto, qty8,
            ROW_NUMBER() OVER (PARTITION BY pdv ORDER BY qty8 DESC) AS rn
        FROM rec
    )
    SELECT pdv, produto
    FROM ranked
    WHERE rn <= 50
""")

# 3) Cross com semanas 1..5 (Jan/2023) e quantidade inicial 0
OUT_PARQUET = "submission.parquet"
con.execute(f"""
    WITH weeks AS (
        SELECT * FROM (VALUES (1),(2),(3),(4),(5)) AS t(semana)
    )
    SELECT weeks.semana, r.pdv, r.produto, 0::INT AS quantidade
    FROM recent_top r
    CROSS JOIN weeks
""")
sub = con.fetch_df()

print(sub.shape)
sub.head()
sub.to_parquet(OUT_PARQUET)

(1331585, 4)


In [3]:
from google.colab import files
files.download('submission.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
!wc -l submission.parquet

9271 submission.parquet
